In [ ]:
image_dir = "../../data/img/06162025"

In [ ]:
import os
from PIL import Image
import pytesseract
import pillow_heif

TARGET_MAX_DIMENSION = 1000  # Resize long edge to 1000px
TARGET_DPI = (300, 300)

def preprocess_image(image, label=""):
    original_size = image.size
    print(f"[{label}] Original size: {original_size}")

    # Remove alpha if present
    if image.mode in ("RGBA", "LA"):
        image = image.convert("RGB")

    # Resize if larger than target
    width, height = image.size
    max_dim = max(width, height)
    if max_dim > TARGET_MAX_DIMENSION:
        scale = TARGET_MAX_DIMENSION / max_dim
        new_size = (int(width * scale), int(height * scale))
        image = image.resize(new_size, Image.LANCZOS)
        print(f"[{label}] Resized to: {new_size}")
    else:
        print(f"[{label}] No resize needed.")

    return image

def auto_rotate_text_image(image_path, save_path=None):
    image = Image.open(image_path)

    try:
        osd = pytesseract.image_to_osd(image)
        rotation_angle = int([line for line in osd.split('\n') if 'Rotate' in line][0].split(':')[1].strip())
        print(f"[{os.path.basename(image_path)}] Detected rotation: {rotation_angle}°")

        if rotation_angle != 0:
            image = image.rotate(-rotation_angle, expand=True)
            print(f"[{os.path.basename(image_path)}] Rotated image by {-rotation_angle}°")

        image = preprocess_image(image, label=os.path.basename(image_path))

        if save_path is None:
            save_path = image_path

        image.save(save_path, "JPEG", dpi=TARGET_DPI)
        print(f"[{os.path.basename(image_path)}] Saved corrected image to {save_path}")

    except Exception as e:
        print(f"[{os.path.basename(image_path)}] Error rotating image: {e}")

def process(directory):
    for filename in os.listdir(directory):
        path = os.path.join(directory, filename)

        if filename.lower().endswith(".heic"):
            jpg_filename = os.path.splitext(filename)[0] + ".jpg"
            jpg_path = os.path.join(directory, jpg_filename)

            try:
                heif_file = pillow_heif.read_heif(path)
                image = Image.frombytes(
                    heif_file.mode,
                    heif_file.size,
                    heif_file.data,
                    "raw"
                )

                image = preprocess_image(image, label=filename)
                image.save(jpg_path, "JPEG", dpi=TARGET_DPI)
                print(f"[{filename}] Converted to {jpg_filename}")

                # Delete the original HEIC file
                os.remove(path)
                print(f"[{filename}] Deleted original HEIC file")

                path = jpg_path  # Continue processing JPG version
            except Exception as e:
                print(f"[{filename}] Failed to convert: {e}")
                continue  # Skip rotation if HEIC conversion fails

        auto_rotate_text_image(path)


In [ ]:
process(image_dir)